In [ ]:
# =============================================================================
# USER CONFIG: edit these five values for normal runs
# =============================================================================

# SAMPLE_START: zero-based first row index in test.csv.
# Example: SAMPLE_START = 0 starts from the first test row.
# Example: SAMPLE_START = 100 starts from row index 100, visually the 101st CSV row after header.
SAMPLE_START = 0

# SAMPLE_COUNT: number of test.csv rows to solve, starting at SAMPLE_START.
# Example: SAMPLE_COUNT = 1 runs a fast smoke test.
# Example: SAMPLE_COUNT = 1001 runs all current test rows.
SAMPLE_COUNT = 1

# GLOBAL_BEAM_WIDTH: total beam width across all GPUs.
# Larger beam usually improves solution probability, but requires larger static capacity and more time.
# Common examples: 2**12, 2**14, 2**16, 2**18.
GLOBAL_BEAM_WIDTH = 2**12

# B_MICRO: inference microbatch size per inference lane.
# Example: 8192 is a stable 2xT4 default; reduce this value if inference memory pressure appears.
B_MICRO = 8192

# BETA: static capacity multiplier for next_state_pool and hash buffers.
# Larger beams can need larger BETA because candidates before prune can exceed final beam size.
# Examples: 1.20 for beam=2**18 after tuning; 32.0 for small smoke tests with wide safety margin on T4.
BETA = 32.0


In [ ]:
# =============================================================================
# ADVANCED CONFIG: edit only when changing runtime behavior or debugging
# =============================================================================

# MAX_DEPTH: maximum beam-search depth.
# Example: 100 is the normal upper bound for full attempts.
MAX_DEPTH = 100

# INFERENCE_PARALLELISM: number of CUDA inference lanes in Stream1.
# Current code loads one TorchScript scorer weight set per rank and reuses it across lanes.
# Example: 1 uses less activation/ring memory; 2 gives better overlap on T4.
INFERENCE_PARALLELISM = 2

# K_EXPAND_TILE: tile size for Stream2 expand/dedup/prune work.
# Example: 16384 is the current T4 default; 8192 gives more frequent threshold/prune updates.
K_EXPAND_TILE = 16384

# SCORE_RING_DEPTH: number of score buffers between inference and expand stages.
# Example: 8 is the current T4 default.
SCORE_RING_DEPTH = 8

# NET_RING_DEPTH: number of NCCL send/recv buffer slots for Stream3.
# Example: 2 is the current T4 default.
NET_RING_DEPTH = 2

# BUCKET_CAP_PER_PEER: fixed candidate-exchange capacity per peer per tile.
# Example: 65536 gives a broad safety margin against bucket overflow in debug runs.
BUCKET_CAP_PER_PEER = 65536

# HASH_LOAD_FACTOR: controls hash-table size; lower value means larger hash table and lower probe pressure.
# Example: 0.35 is conservative for small-beam tests; 0.45 is a tighter tuned value.
HASH_LOAD_FACTOR = 0.35

# PROBE_LIMIT: maximum linear probes in the GPU hash table.
# Example: 512 is conservative; 256 may be faster but can raise overflow risk.
PROBE_LIMIT = 512

# HISTORY_BACKEND: history storage backend.
# gpu = current/legacy GPU history mode.
# cpu = GPU keeps only current transition layer, CPU stores transition archive in static RAM arrays.
HISTORY_BACKEND = 'cpu'

# CPU_HISTORY_CHECKPOINT: write CPU history archive and frontier checkpoint after each complete depth.
# 0 = no disk checkpoint; 1 = write resumable checkpoint files.
CPU_HISTORY_CHECKPOINT = 0

# RESUME_BEAMSEARCH: restore latest complete CPU-history checkpoint before continuing search.
# Requires HISTORY_BACKEND='cpu' and CPU_HISTORY_CHECKPOINT=1 from the previous run.
RESUME_BEAMSEARCH = 0

# RESUME_SUBMISSION: skip already solved ids already present in submission.csv.
# Example: 1 continues a previous partial submission file after a crash or Kaggle timeout.
RESUME_SUBMISSION = 0

# CPU_HISTORY_WORKERS: worker threads for CPU archive/checkpoint work.
# Example: 2 is enough for T4 runs because GPU work dominates runtime.
CPU_HISTORY_WORKERS = 2

# LOG_EVERY: seconds between notebook wrapper heartbeat logs.
# Example: 30 prints wrapper progress while long subprocesses run.
LOG_EVERY = 30

# BEAM_DEBUG: compile-time C++/CUDA debug variant selector.
# 0 = release extension: BEAM_DEBUG_ON=0, debug C++ code is excluded by #if.
# 1 = debug extension: BEAM_DEBUG_ON=1, optional engine/depth diagnostics can run.
BEAM_DEBUG = 0

# DEPTH_LOG_EVERY: depth-result log interval, active only when BEAM_DEBUG=1.
# 0 = no depth logs. Example: 1 logs every depth in debug runs.
DEPTH_LOG_EVERY = 0

# SAMPLE_LOG_EVERY: rank0 progress log interval after completed samples.
# 1 = log every solved/attempted sample; 10 = log every tenth sample; 0 = suppress sample progress logs.
SAMPLE_LOG_EVERY = 1

# TIMEOUT_SEC: maximum wall time for the solver subprocess.
# Example: 0 disables timeout; 43200 allows a 12-hour Kaggle session.
TIMEOUT_SEC = 0

# GITHUB_REPO_URL: project code source. Kaggle clones this repo at run time.
# Push code to GitHub first, then rerun the notebook.
GITHUB_REPO_URL = 'https://github.com/TryDotAtwo/MultiGPUBeamSearch.git'

# GITHUB_BRANCH: Git branch or tag used by Kaggle clone.
# Example: master for current production source; a commit hash is safer for exact reproducibility.
GITHUB_BRANCH = 'master'

# PROJECT_DIR: clone destination inside Kaggle working directory.
PROJECT_DIR = '/kaggle/working/CayleyBeam100H100'

# MODEL_DATASET_HINT: substring used to find the Kaggle dataset that contains model weights.
# The project code comes from GitHub; this dataset should contain only weights/assets not stored in Git.
MODEL_DATASET_HINT = 'cayleybeam-fullbeamnice-project'

# SCORER_INIT_PY: optional path to a custom scorer initializer Python file.
# Empty string uses the default FullBeamNice scorer from MODEL_DATASET_HINT.
# Example: '/kaggle/input/my-model/init_scorer.py'
SCORER_INIT_PY = ''

# SUBMISSION_PATH: output CSV written incrementally after each sample.
SUBMISSION_PATH = '/kaggle/working/submission.csv'


In [ ]:
# =============================================================================
# CUSTOM SCORER API: how to plug in your own neural model or heuristic
# =============================================================================

# Solver consumes only canonical action scores shaped [B, 24].
# Current action order is:
# [-B, -BL, -BR, -D, -DL, -DR, -F, -FL, -FR, -L, -R, -U,
#   B,  BL,  BR,  D,  DL,  DR,  F,  FL,  FR,  L,  R,  U]
# Larger score means better candidate.

# To use a custom model, put an initializer file into a Kaggle dataset and set SCORER_INIT_PY above.
# The initializer should return a Torch module/callable plus metadata.
# Export adapter converts initializer output to canonical [B, 24], then C++ hot path sees only [micro_size, 24].
# Beam kernels contain no variable-output branches.

# Supported output_kind values:
# 1. output_kind='action24'
#    Initializer returns direct action scores [B, 24]. Optional metadata can remap action names.
#
# 2. output_kind='action12'
#    Initializer returns named 12-move scores. Adapter expands/remaps scores into the current 24-action order.
#
# 3. output_kind='value1_after_move'
#    Initializer returns scalar value [B, 1]. Adapter applies 24 action permutations in TorchScript,
#    evaluates child states, and reshapes child values to [B, 24].
#
# 4. output_kind='heuristic24'
#    Initializer returns a TorchScriptable heuristic module, for example Hamming/Bellman-style scoring.
#    Output must be convertible to [B, 24].

# Minimal initializer sketch:
#
# import torch
#
# def create_scorer():
#     model = MyTorchModule()
#     model.eval()
#     return model, {
#         'output_kind': 'action24',
#         'higher_is_better': True,
#         'action_order': ['-B','-BL','-BR','-D','-DL','-DR','-F','-FL','-FR','-L','-R','-U',
#                          'B','BL','BR','D','DL','DR','F','FL','FR','L','R','U'],
#     }

# TODO: generalize solver/export path to any generator count, not only 24 actions.
# Current GPU kernels, transition records, and scorer adapter are specialized for 24 actions.
print('CUSTOM_SCORER_DOC_READY: set SCORER_INIT_PY to use a custom Torch scorer')


In [ ]:
# =============================================================================
# YANDEX CLOUD TODO: future Kaggle -> Yandex Cloud bridge
# =============================================================================

# Current state: all computation runs inside Kaggle only.
# TODO: sync checkpoints, submission.csv, and logs to Yandex Object Storage.
# TODO: read credentials from Kaggle Secrets, never from notebook source code.
# TODO: add resume flow where Kaggle can restart from the latest cloud checkpoint.
# TODO: add optional offload path from Kaggle smoke tests to Yandex 2xA100 or larger clusters.
# No cloud code is active in this cell by design.
print('YANDEX_CLOUD_STATUS: disabled; current_run_location=Kaggle_only')


In [ ]:
# =============================================================================
# RUN SOLVER: clone code from GitHub, copy competition input files, use model dataset only for weights
# =============================================================================

import csv
import json
import os
import pathlib
import queue
import shutil
import subprocess
import sys
import threading
import time
from datetime import datetime

KAGGLE_INPUT = pathlib.Path('/kaggle/input')
PROJECT_PATH = pathlib.Path(PROJECT_DIR)


def now():
    return datetime.now().strftime('%H:%M:%S')


def gpu_snapshot():
    try:
        out = subprocess.check_output([
            'nvidia-smi', '--query-gpu=index,name,memory.used,memory.total,utilization.gpu', '--format=csv,noheader,nounits'
        ], text=True, stderr=subprocess.STDOUT).strip()
        return out.replace('\n', ' | ')
    except Exception as exc:
        return f'nvidia-smi-unavailable:{type(exc).__name__}:{exc}'


def run_live(cmd, *, env=None, cwd=None, title='', timeout_sec=None, heartbeat_sec=30):
    print('\n' + '=' * 96, flush=True)
    print(f'[{now()}] {title}', flush=True)
    print('=' * 96, flush=True)
    print(f'[{now()}] cmd={cmd}', flush=True)
    merged = os.environ.copy()
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    merged.setdefault('PYTHONUNBUFFERED', '1')
    proc = subprocess.Popen(cmd, cwd=str(cwd) if cwd else None, env=merged, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    q = queue.Queue()

    def reader():
        for line in iter(proc.stdout.readline, ''):
            q.put(line.rstrip('\n'))
        proc.stdout.close()

    threading.Thread(target=reader, daemon=True).start()
    start = time.time()
    last_heartbeat = 0.0
    last_output = time.time()
    lines = []
    while True:
        drained = False
        while True:
            try:
                line = q.get_nowait()
            except queue.Empty:
                break
            drained = True
            last_output = time.time()
            lines.append(line)
            print(line, flush=True)
        rc = proc.poll()
        elapsed = time.time() - start
        if timeout_sec is not None and elapsed > timeout_sec and rc is None:
            proc.kill()
            raise TimeoutError(f'timeout after {elapsed:.1f}s: {cmd}')
        if rc is not None:
            while True:
                try:
                    line = q.get_nowait()
                except queue.Empty:
                    break
                lines.append(line)
                print(line, flush=True)
            print(f'[{now()}] process_exit | returncode={rc} | elapsed_sec={elapsed:.1f}', flush=True)
            text = '\n'.join(lines)
            if rc != 0:
                raise subprocess.CalledProcessError(rc, cmd, output=text)
            return text
        if time.time() - last_heartbeat >= heartbeat_sec:
            print(f'[{now()}] still_running | elapsed_sec={elapsed:.1f} | silent_for_sec={time.time()-last_output:.1f} | gpu=[{gpu_snapshot()}]', flush=True)
            last_heartbeat = time.time()
        time.sleep(0.2 if drained else 0.5)


def find_competition_dir():
    candidates = []
    required = ['puzzle_info.json', 'sample_submission.csv', 'test.csv']
    for p in KAGGLE_INPUT.rglob('*'):
        if p.is_dir() and all((p / name).exists() for name in required):
            candidates.append(p)
    if not candidates:
        listing = [str(x) for x in KAGGLE_INPUT.glob('*')]
        raise FileNotFoundError(f'competition files not found under /kaggle/input; top_level={listing}')
    candidates.sort(key=lambda x: (0 if 'competition' in str(x).lower() else 1, len(str(x))))
    return candidates[0]


def find_model_source():
    candidates = []
    for p in KAGGLE_INPUT.rglob('*'):
        if not p.is_dir():
            continue
        has_model = (p / 'FullBeamNice').exists() or (p / 'FullBeamNice.zip').exists()
        hint_ok = not MODEL_DATASET_HINT or MODEL_DATASET_HINT.lower() in str(p).lower()
        if has_model and hint_ok:
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError('model dataset with FullBeamNice or FullBeamNice.zip not found; attach model dataset')
    candidates.sort(key=lambda x: len(str(x)))
    return candidates[0]


def prepare_project_from_github():
    if PROJECT_PATH.exists():
        shutil.rmtree(PROJECT_PATH)
    run_live(['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, GITHUB_REPO_URL, str(PROJECT_PATH)], title='clone project from GitHub', timeout_sec=600)

    data_dir = PROJECT_PATH / 'data'
    data_dir.mkdir(parents=True, exist_ok=True)
    competition_dir = find_competition_dir()
    for name in ['puzzle_info.json', 'sample_submission.csv', 'test.csv']:
        src = competition_dir / name
        if not src.exists():
            raise FileNotFoundError(f'competition file missing: {src}')
        shutil.copy2(src, data_dir / name)

    model_src = find_model_source()
    if (model_src / 'FullBeamNice').exists():
        shutil.copytree(model_src / 'FullBeamNice', PROJECT_PATH / 'FullBeamNice', dirs_exist_ok=True)
    elif (model_src / 'FullBeamNice.zip').exists():
        shutil.unpack_archive(str(model_src / 'FullBeamNice.zip'), PROJECT_PATH / 'FullBeamNice')
    else:
        raise FileNotFoundError(f'FullBeamNice model source missing inside {model_src}')

    print('PROJECT_SOURCE_GITHUB', GITHUB_REPO_URL, GITHUB_BRANCH, flush=True)
    print('COMPETITION_DATA_DIR', str(competition_dir), flush=True)
    print('MODEL_SOURCE_DIR', str(model_src), flush=True)
    print('PROJECT_READY', str(PROJECT_PATH), flush=True)


prepare_project_from_github()

base_env = {
    'PYTHONUNBUFFERED': '1',
    'CUDA_VISIBLE_DEVICES': '0,1',
    'USE_CUDA_GRAPHS': '1',
    'INFERENCE_BACKEND': 'torchscript_ensemble',
    'INFERENCE_PARALLELISM': INFERENCE_PARALLELISM,
    'HISTORY_BACKEND': HISTORY_BACKEND,
    'CPU_HISTORY_CHECKPOINT': CPU_HISTORY_CHECKPOINT,
    'RESUME_BEAMSEARCH': RESUME_BEAMSEARCH,
    'RESUME_SUBMISSION': RESUME_SUBMISSION,
    'CPU_HISTORY_WORKERS': CPU_HISTORY_WORKERS,
    'GLOBAL_BEAM_WIDTH': GLOBAL_BEAM_WIDTH,
    'MAX_DEPTH': MAX_DEPTH,
    'B_MICRO': B_MICRO,
    'K_EXPAND_TILE': K_EXPAND_TILE,
    'SCORE_RING_DEPTH': SCORE_RING_DEPTH,
    'NET_RING_DEPTH': NET_RING_DEPTH,
    'BUCKET_CAP_PER_PEER': BUCKET_CAP_PER_PEER,
    'BETA': BETA,
    'HASH_LOAD_FACTOR': HASH_LOAD_FACTOR,
    'PROBE_LIMIT': PROBE_LIMIT,
    'TEST_START': SAMPLE_START,
    'TEST_COUNT': SAMPLE_COUNT,
    'SUBMISSION_PATH': SUBMISSION_PATH,
    'SUBMISSION_APPEND_EACH': '1',
    'LOG_EVERY': LOG_EVERY,
    'BEAM_DEBUG': BEAM_DEBUG,
    'DEPTH_LOG_EVERY': DEPTH_LOG_EVERY,
    'SAMPLE_LOG_EVERY': SAMPLE_LOG_EVERY,
    'NCCL_IB_DISABLE': '1',
    'NCCL_P2P_DISABLE': '1',
    'NCCL_SOCKET_IFNAME': 'lo',
    'GLOO_SOCKET_IFNAME': 'lo',
    'NCCL_DEBUG': 'WARN',
    'BUILD_VERBOSE': '0',
}
if SCORER_INIT_PY:
    base_env['SCORER_INIT_PY'] = SCORER_INIT_PY

run_live([sys.executable, '-u', 'scripts/t4_sizing.py'], env=base_env, cwd=PROJECT_PATH, title='static memory sizing', timeout_sec=120)
solver_output = run_live([
    sys.executable, '-u', '-m', 'torch.distributed.run', '--standalone', '--nnodes=1', '--nproc_per_node=2', 'scripts/solve_testcsv_2gpu.py'
], env=base_env, cwd=PROJECT_PATH, title='2xT4 beam search', timeout_sec=(None if TIMEOUT_SEC <= 0 else TIMEOUT_SEC))

print('RUN_DONE', json.dumps({
    'submission_path': SUBMISSION_PATH,
    'sample_start': SAMPLE_START,
    'sample_count': SAMPLE_COUNT,
    'global_beam_width': GLOBAL_BEAM_WIDTH,
    'history_backend': HISTORY_BACKEND,
}, ensure_ascii=False), flush=True)


In [ ]:
# =============================================================================
# METRICS: summary plus solved-length histogram
# =============================================================================

from pathlib import Path
import csv
import math
import statistics
from collections import Counter

submission_path = Path(SUBMISSION_PATH)
if not submission_path.exists():
    raise FileNotFoundError(f'submission file not found: {submission_path}')

lengths_all = []
solved_lengths = []
rows = []
with submission_path.open('r', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        path = (row.get('path') or '').strip()
        length = 0 if path == '' else len(path.split('.'))
        rows.append(row)
        lengths_all.append(length)
        if length > 0:
            solved_lengths.append(length)

total_count = len(rows)
unsolved_count = total_count - len(solved_lengths)
solved_percent = 0.0 if total_count == 0 else 100.0 * len(solved_lengths) / total_count
total_len = sum(lengths_all)
mean_len_all = None if not lengths_all else statistics.mean(lengths_all)
median_len_all = None if not lengths_all else statistics.median(lengths_all)
max_len_solved = None if not solved_lengths else max(solved_lengths)
min_len_solved = None if not solved_lengths else min(solved_lengths)
mean_len_solved = None if not solved_lengths else statistics.mean(solved_lengths)
median_len_solved = None if not solved_lengths else statistics.median(solved_lengths)

summary = {
    'total_count': total_count,
    'unsolved_count': unsolved_count,
    'solved_percent': solved_percent,
    'total_len': total_len,
    'mean_len_all': mean_len_all,
    'median_len_all': median_len_all,
    'max_len_solved': max_len_solved,
    'min_len_solved': min_len_solved,
    'mean_len_solved': mean_len_solved,
    'median_len_solved': median_len_solved,
    'solved_lengths': solved_lengths,
}
print('METRICS_SUMMARY')
print(json.dumps(summary, ensure_ascii=False, indent=2))

hist = Counter(solved_lengths)
print('SOLVED_LENGTH_HISTOGRAM')
if not hist:
    print('(empty)')
else:
    max_count = max(hist.values())
    for length in sorted(hist):
        bar = '#' * max(1, round(50 * hist[length] / max_count))
        print(f'{length:4d} | {hist[length]:5d} | {bar}')


In [ ]:
# =============================================================================
# SUBMIT: upload submission.csv to the competition
# =============================================================================

submit_message = (
    f"test run: "
    f"beam={GLOBAL_BEAM_WIDTH} "
    f"depth={MAX_DEPTH} "
    f"samples={SAMPLE_START}..{SAMPLE_START + SAMPLE_COUNT - 1} "
    f"parallelism={INFERENCE_PARALLELISM} "
    f"b_micro={B_MICRO} "
    f"k_tile={K_EXPAND_TILE} "
    f"beta={BETA}"
)

print("SUBMIT_MESSAGE:", submit_message)

!kaggle competitions submit \
  -c cayley-py-megaminx \
  -f /kaggle/working/submission.csv \
  -m "$submit_message"
